# Heart-rate branch -- Stage B: the corpora

Loads and audits the two heart-rate corpora. **No model is trained here and none
should be.** Stage B answers one question: do WESAD and PPG-DaLiA contain what
the plan assumes they contain, in the shape the device will see?

That question comes first because the companion repository's pipeline did not
ask it. It trained and exported a four-class model whose `gunshot` class held
zero samples (`MULTIMODAL_PLAN.md`, F3). Nothing was wrong with its training
code. What was missing was a count, before the training, of what the data
actually held. Section 4 below is that count, and it raises rather than prints.

## What arrives here

| | WESAD | PPG-DaLiA |
|---|---|---|
| role | the positives | the negatives, and the false-alarm denominator |
| subjects | 15 (S2-S17, no S1/S12) | 15 |
| labels | baseline / stress / amusement / meditation | 8 everyday activities |
| ground-truth HR | derived here from a 700 Hz chest ECG | shipped, ECG-derived |
| size | ~18 GB | ~1.6 GB |

Both are Empatica E4 wrist recordings: BVP at 64 Hz, ACC at 32 Hz. Both are
resampled here to **100 Hz, the MAX30102's rate**, so that `hr.py` in Stage C is
developed at exactly the rate it will run at on the device rather than leaving a
64 -> 100 Hz conversion as an untested step at deployment.

## Three caveats that belong on every number derived from this

**The stressor is a job interview.** WESAD's stress condition is the Trier
Social Stress Test -- public speaking and mental arithmetic in front of a panel.
It is the closest public proxy available and it is not an assault.

**An E4 is not a MAX30102.** Different LED wavelength, adaptive gain, a properly
tensioned strap. The target is a breakout board on veroboard in a hand-made
mount. This is the heart-rate equivalent of Stage 0's wrist/waist gap.

**Every DaLiA window is labelled non-stress by assumption, not by observation.**
DaLiA runs no stress protocol. Someone stuck in traffic during the driving block
may well have been stressed. That assumption is exactly what makes the corpus
usable as a false-alarm denominator, so it is stated rather than buried.

## 1. Setup

On Colab: clone the repo and install it editable. Unlike `01_movement.ipynb`,
nothing here needs TensorFlow -- the loaders, the ECG reduction and the gate are
NumPy and SciPy only, which is deliberate. Stage B should be runnable, and
re-runnable, without a GPU runtime.

In [ ]:
REPO_URL = "https://github.com/Mursalin1011/shahoshi-model.git"

import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
_SECRET: list[str] = []          # token text to scrub from any output


def _run(cmd, cwd=None):
    """Run a command and surface its stderr on failure.

    check_call() raises with only an exit code, which is how a plain
    'repository not found' turned into an opaque CalledProcessError: exit 128.
    """
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.returncode != 0:
        detail = (p.stderr or p.stdout or "").strip()
        shown = " ".join(cmd)
        for s in _SECRET:                     # never echo a token back
            detail, shown = detail.replace(s, "***"), shown.replace(s, "***")
        raise RuntimeError(f"$ {shown}\nexit {p.returncode}\n{detail}")
    return p.stdout


def _auth_url():
    """Add a token from Colab Secrets, if one is configured.

    Read from the Secrets store rather than typed into a cell: a token pasted
    into a notebook is saved inside the .ipynb and travels to anyone the
    notebook is shared with.
    """
    if not IN_COLAB:
        return REPO_URL
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        return REPO_URL                        # no secret set, or access not granted
    if not token:
        return REPO_URL
    _SECRET.append(token)
    return REPO_URL.replace("https://", f"https://oauth2:{token}@")


def _find_root():
    """Nearest directory at or above cwd that is this repo. None if outside it."""
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "src" / "shahoshi" / "__init__.py").exists():
            return d
    return None


PRIVATE_REPO_HELP = """
git could not read the repository.

Colab clones anonymously, so a PRIVATE repo fails here even though it works from
your own machine. Two ways forward:

  1. Make the repo public -- Settings -> General -> Change visibility.
     Note first that the proposal PDFs committed at the repo root carry student
     names and email addresses, which publishing would expose.

  2. Keep it private and give Colab a token:
       a. GitHub -> Settings -> Developer settings -> Personal access tokens
          -> Fine-grained tokens. Scope it to this one repo, Contents: Read-only.
       b. In Colab, click the key icon in the left sidebar, add a secret named
          GITHUB_TOKEN, paste the token there, and enable notebook access.
       c. Re-run this cell.
     Do not paste the token into a notebook cell -- it would be saved in the
     .ipynb file.
"""

# --- 1. get the code -------------------------------------------------------
root = _find_root()

if root is None:
    if not IN_COLAB:
        raise SystemExit(
            "Run this notebook from inside a checkout of the repo "
            "(expected to find src/shahoshi/ at or above the working directory)."
        )
    if not REPO_URL:
        raise SystemExit("Set REPO_URL to your GitHub repo before running on Colab.")

    name = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    try:
        if Path(name).exists():
            _run(["git", "fetch", "--depth", "1", "origin", "main"], cwd=name)
            _run(["git", "reset", "--hard", "origin/main"], cwd=name)
            print(f"updated existing checkout: {name}")
        else:
            _run(["git", "clone", "--depth", "1", _auth_url(), name])
            print(f"cloned: {name}")
    except RuntimeError as exc:
        text = str(exc).lower()
        if any(s in text for s in ("not found", "authentication failed",
                                   "could not read", "403", "401", "terminal prompts")):
            raise SystemExit(PRIVATE_REPO_HELP + f"\ngit said:\n{exc}") from None
        raise

    root = Path(name).resolve()

os.chdir(root)

# --- 2. make the package importable ---------------------------------------
# Two mechanisms, deliberately. `pip install -e .` is the tidy one, but on a
# long-lived Colab kernel the import finder it drops into site-packages is not
# always picked up by the already-running interpreter -- which surfaces as
# ModuleNotFoundError several cells later, far from the cause. Putting src/ on
# sys.path is what actually guarantees importability, so the install is treated
# as best-effort and the path entry is unconditional.
try:
    _run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    print("editable install ok")
except RuntimeError as exc:
    print(f"editable install failed (continuing via sys.path)\n{exc}\n")

src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# --- 3. verify here, not three cells from now -----------------------------
try:
    import shahoshi
except ModuleNotFoundError as exc:
    raise SystemExit(
        f"cannot import shahoshi even with {src} on sys.path.\n"
        f"  cwd            : {Path.cwd()}\n"
        f"  src exists     : {Path(src).exists()}\n"
        f"  package exists : {(Path(src) / 'shahoshi' / '__init__.py').exists()}\n"
        f"  error          : {exc}\n"
        "If the package directory is missing, the clone is incomplete -- delete "
        "the checkout directory and re-run this cell."
    ) from None

print(f"shahoshi {shahoshi.__version__} from {Path(shahoshi.__file__).parent}")
print(f"repo root: {Path.cwd()}")
print(f"in colab : {IN_COLAB}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from shahoshi import datasets
from shahoshi.datasets import availability, cache, dalia, e4, wesad
from shahoshi.datasets.base import HR_CHANNELS, HR_FS

print(f"device rate     : {HR_FS} Hz")
print(f"channels        : {list(HR_CHANNELS)}")
print(f"window / hop    : {e4.WIN_SECONDS:.0f} s / {e4.STRIDE_SECONDS:.0f} s "
      f"({e4.win_samples(HR_FS)} / {e4.stride_samples(HR_FS)} samples)")
print(f"WESAD subjects  : {len(wesad.SUBJECTS)}")
print(f"DaLiA subjects  : {len(dalia.SUBJECTS)}")

## 2. Where the data goes

Two locations, and the difference matters on Colab.

`DATA_DIR` holds the downloaded archives and their extraction. It lives on the
**session disk**, which is wiped when the runtime recycles. WESAD alone is ~18 GB
extracted, so this is not something to put on Drive.

`CACHE_DIR` holds the *distilled* form: per subject, just BVP and ACC resampled
to 100 Hz, a condition code, and the ground-truth HR series. That is a few
hundred MB for both corpora, and it is what makes the second run cheap -- point
it at Drive and the 18 GB download never happens again.

Mounting Drive is optional. Without it everything still works; it just re-parses
the archives every session.

In [ ]:
USE_DRIVE = True          # set False to keep everything on the session disk

DATA_DIR = "/content/data"
CACHE_DIR = None          # None -> cache.default_root() picks Drive if mounted

if USE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except ImportError:
        print("not on Colab -- using local paths")
        DATA_DIR = "./data"

print(cache.describe(cache.default_root(CACHE_DIR)))

## 3. Load DaLiA first

DaLiA is 1.6 GB against WESAD's 18 GB, so it exercises the entire path --
download, extract, unpickle, resample, window, reference alignment -- in a few
minutes instead of most of an hour. If something is going to fail, it fails here
cheaply.

In [ ]:
hr_neg = datasets.load_hr(DATA_DIR, tags=["dalia"], cache_dir=CACHE_DIR)

### The gate on DaLiA alone must fail

Run it now, before WESAD, precisely because it **should** report a fatal failure:
DaLiA contains no stress windows at all, so there is nothing to detect. A gate
that passed here would be a gate that is not checking anything.

`report()` prints and returns; `require()` is the one that raises. This cell
calls the printing one on purpose.

In [ ]:
print(availability.report(hr_neg))

## 4. Load WESAD and run the real gate

This is the ~18 GB download on a cold run. On a warm one, with `CACHE_DIR`
pointing at Drive, it reads a few hundred MB of `.npz` instead.

`with_reference=True` reduces each subject's 700 Hz chest ECG to a beat series
by Pan-Tompkins R-peak detection. That is the slow half of a cold run and it is
what gives every window a ground-truth heart rate to be scored against in
Stage C. The ECG is **truth only** -- it is never a model input, and the device
does not have a chest strap.

In [ ]:
hr = datasets.load_hr(
    DATA_DIR,
    tags=["dalia", "wesad"],
    cache_dir=CACHE_DIR,
    with_reference=True,
)

In [ ]:
print(availability.report(hr))
availability.require(hr)         # raises if any fatal check failed

### What the fatal checks are for

| check | what it prevents |
|---|---|
| positives exist | defect F3: a class with no samples that trains and exports anyway |
| positives span subjects | positives confined to one person, so no subject-disjoint split exists and the reported separation is one physiology |
| acc scaling | the E4 stores acceleration in 1/64 g. Left unscaled it loads, windows and trains, and is wrong by a factor of 64. The gate asserts that a wrist at rest reads ~1 g rather than trusting the constant |
| bvp varies | a flat PPG channel, which is what a mis-keyed pickle produces |
| subject ids unique | WESAD and DaLiA both number subjects `S1`, `S2`, ... Without the corpus prefix a split would leak the same id across both |

## 5. Look at the signals

Two plots, both of which are checks rather than decoration.

The first is one window of BVP per condition. A PPG at rest is visibly periodic
at roughly 1 Hz; if these look like noise, the resampling or the channel order is
wrong and every number after this is meaningless.

The second is the ground-truth HR distribution by condition. WESAD's stress block
should sit visibly above its baseline block. If it does not, either the ECG
reduction or the label alignment is broken -- and that is worth knowing now,
rather than after a deviation rule fails to separate anything in Stage C.

In [ ]:
conds = sorted(set(hr.condition.tolist()))
fig, axes = plt.subplots(1, len(conds), figsize=(3.2 * len(conds), 2.6), sharey=True)
t = np.arange(hr.win) / hr.fs
for ax, cond in zip(np.atleast_1d(axes), conds):
    i = int(np.flatnonzero(hr.condition == cond)[0])
    ax.plot(t, hr.X[i, :, 0], lw=0.8)
    ax.set_title(f"{cond}\n{hr.subject[i]}", fontsize=9)
    ax.set_xlabel("s")
np.atleast_1d(axes)[0].set_ylabel("BVP")
plt.suptitle("One BVP window per condition -- these must look periodic")
plt.tight_layout()
plt.show()

In [ ]:
ref = hr.has_reference
labelled = [c for c in conds if (ref & (hr.condition == c)).sum() > 20]
data = [hr.hr_ref[ref & (hr.condition == c)] for c in labelled]

plt.figure(figsize=(1.3 * len(labelled) + 3, 4))
plt.boxplot(data, labels=labelled, showfliers=False)
plt.ylabel("ground-truth HR (bpm)")
plt.title("Reference HR by condition -- stress should sit above baseline")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

for c in labelled:
    v = hr.hr_ref[ref & (hr.condition == c)]
    print(f"  {c:<14s} n={len(v):>7,}  median {np.median(v):6.1f} bpm  "
          f"IQR {np.percentile(v, 25):.1f}-{np.percentile(v, 75):.1f}")

## 6. Motion, which is the thing that will break the PPG

The accelerometer is in `HR_CHANNELS` only as an artefact gate: a wrist PPG
degrades under movement, and Stage C needs to know when to distrust its own
estimate. This is the first look at how much motion each condition carries.

Note that `acc_*` here keeps gravity, unlike the movement branch's channels,
which have it removed. That is why a still wrist reads ~1 g rather than ~0.

In [ ]:
mag = np.linalg.norm(hr.X[:, :, 1:], axis=2)
motion = mag.std(axis=1)          # within-window variation, so gravity drops out

for c in conds:
    m = hr.condition == c
    print(f"  {c:<14s} n={int(m.sum()):>7,}  "
          f"median |acc| {np.median(mag[m]):5.2f} g  "
          f"motion {np.median(motion[m]):6.3f} g")

## 7. Where this leaves Stage C

The gate has said whether the corpora hold what the plan assumed. What it cannot
say is whether a **frozen, untrained deviation statistic** -- a z-score against a
rolling personal baseline -- already separates stress from ordinary activity.
That is Stage C, and it is deliberately the next step rather than a model.

The argument for trying a statistic before a model is the same one that dropped
the `lay` class in Stage 0: a z-score against a personal baseline transfers
across sensors far better than anything fitted to E4 amplitudes, and the gap
between an E4 and a MAX30102 on veroboard is the largest uncontrolled variable
in this branch. If a statistic captures most of the separation, fitting a model
to E4 amplitudes buys a number that will not survive the hardware change.

Stage C also owns the measurement this notebook sets up but does not make: the
error of the PPG-derived HR against the `hr_ref` loaded here. That is the
estimator's own accuracy, separate from whether the estimate separates stress.